# Classifying `[CLS]` embeddings with a logistic regression

**Scenario: routing a new case.** A hospital receives case notes and wants
to send each one to the right team. We train a classifier on short labeled
medical sentences, then ask it where the four MultiCaRe texts of
`../embedding/text-embeddings.ipynb` should go.

* **Data:** the titles of PubMed abstracts from the *Medical Abstracts*
  corpus (Schopf et al., 2022), labeled with the patient's condition. We use
  four classes: **neoplasms, digestive, nervous, cardiovascular** (the
  fifth class of the corpus, "general pathological conditions", is a
  catch-all and is left out).
* **Features:** the `[CLS]` vector of `BAAI/bge-small-en-v1.5`, the same
  model as in the embedding notebook (384 numbers per title). The model is
  **frozen**: it is used, never trained.
* **Classifier:** a logistic regression, the only part that learns.

## 1. Preparation

Model, the four MultiCaRe texts, and the data. `load_titles()`
(in `classification_helpers.py`) downloads the corpus once into `data/` and
cleans it: it keeps abstracts with a single label, cuts each one at the
end of its title, and drops titles longer than 30 words (titles without a
final period, where the cut falls inside the abstract).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from classification_helpers import load_titles

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
pd.set_option("display.max_colwidth", 200)

MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()

TEXTS = {
    "Text 1": "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant "
              "abdominal pain associated with nausea and constipation.",
    "Text 2": "A 41-year-old female with a history of cesarean section presented to the emergency "
              "department with continuous abdominal pain for 12 h, localized in the right iliac fossa (RIF).",
    "Text 3": "A 23-year-old male was brought to the emergency of Hamad General Hospital in Doha, after "
              "being found on the floor dizzy and confused, with urinary incontinence although no one "
              "witnessed convulsions.",
    "Text 4": "However, 10 days later, she developed a non-pruritic morbilliform rash spreading from trunk "
              "to the entire body, including palms, soles, and face with mild lip and eye swelling.",
}

titles = load_titles()

14,438 rows → 11,227 distinct abstracts → 8,298 with a single label → 5,683 titles in 4 classes


## 2. The data

How many titles each class has, and a few of them.

In [2]:
print(titles.label.value_counts(), "\n")
for label, group in titles.groupby("label"):
    print(f"── {label}")
    for t in group.title.sample(3, random_state=0):
        print("  ", t)

label
neoplasms         2130
cardiovascular    1862
nervous           1015
digestive          676
Name: count, dtype: int64 

── cardiovascular
   Cardiac arrest due to spontaneous coronary artery dissection in a patient with coronary ectasia--a case report.
   Clinical association of autoantibodies to fibrillarin with diffuse scleroderma and disseminated telangiectasia.
   Circadian rhythms in blood pressure in school-age children of normotensive and hypertensive parents.
── digestive
   Gender differences in Manning criteria in the irritable bowel syndrome.
   Comparison of gall bladder bile and endoscopically obtained duodenal bile.
   Sulphasalazine and prednisone compared with sulphasalazine for treating active Crohn disease.
── neoplasms
   Effects of epidermal growth factor and analogues of luteinizing hormone-releasing hormone and somatostatin on phosphorylation and dephosphorylation of tyrosine residues of specific protein substrates in various tumors.
   Different effect of b

The classes have different sizes. We take the **same number of titles
from each class** (600, a bit less than the smallest class), so that a
classifier cannot win by always answering the biggest class. Then we split
them: **two thirds for training, one third for testing**. The test titles
are never shown to the classifier during training: they measure how it
does on titles it has not seen.

In [3]:
PER_CLASS = 600

sample = titles.groupby("label").sample(PER_CLASS, random_state=0)
train, test = train_test_split(sample, test_size=1/3, stratify=sample.label, random_state=0)
print("train:", len(train), "titles   test:", len(test), "titles")

train: 1600 titles   test: 800 titles


## 3. Many texts at once: padding and the attention mask

In the embedding notebook each text went through the model alone. With
1,600 titles we send them in **batches** (64 at a time), which is much
faster. A batch is a rectangle of token ids, so the shorter texts are
**padded** with `[PAD]` up to the longest one, and the **attention mask**
(1 = real token, 0 = padding) tells the model to ignore the padding.
`[CLS]` is still at position 0 of every row.

Two titles in one batch:

In [4]:
batch = tokenizer(["Diarrhea.", "Memory complaints in older adults."],
                  padding=True, return_tensors="pt")

for ids, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print("tokens:", tokenizer.convert_ids_to_tokens(ids))
    print("mask:  ", mask.tolist())

tokens: ['[CLS]', 'dia', '##rr', '##hea', '.', '[SEP]', '[PAD]', '[PAD]']
mask:   [1, 1, 1, 1, 1, 1, 0, 0]
tokens: ['[CLS]', 'memory', 'complaints', 'in', 'older', 'adults', '.', '[SEP]']
mask:   [1, 1, 1, 1, 1, 1, 1, 1]


The same three steps as before (tokenize, encode, keep `[CLS]`), now for a
whole batch at a time:

In [5]:
def embed(texts, batch_size=64):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size],
                          padding=True, return_tensors="pt")      # 1. texts → padded token ids + mask
        with torch.no_grad():
            outputs = model(**batch)                              # 2. ids → one vector per token
        vectors.append(outputs.last_hidden_state[:, 0])           # 3. [all texts, position 0] = [CLS]
    return torch.cat(vectors).numpy()

start = time.time()
X_train = embed(train.title.tolist())
X_test = embed(test.title.tolist())
print(f"X_train: {X_train.shape}   X_test: {X_test.shape}   ({time.time() - start:.0f} s)")

X_train: (1600, 384)   X_test: (800, 384)   (25 s)


Does the padding change anything? The `[CLS]` vector of text 1 computed
alone and computed in a batch with the three longer texts should be the same
(up to rounding):

In [6]:
alone = embed([TEXTS["Text 1"]])[0]
in_batch = embed(list(TEXTS.values()))[0]
print("largest difference:", abs(alone - in_batch).max())

largest difference: 7.1525574e-07


## 4. Training

A logistic regression with four classes learns **one weight per dimension
for each class**, plus one bias per class. For a title it computes four
scores (weighted sums of its 384 numbers) and turns them into four
probabilities that add up to 1 (softmax). Training adjusts the weights so
that the right class gets a high probability on the training titles.

In [7]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, train.label)

print("classes:", list(classifier.classes_))
print("weights:", classifier.coef_.shape, "  biases:", classifier.intercept_.shape)

classes: ['cardiovascular', 'digestive', 'neoplasms', 'nervous']
weights: (4, 384)   biases: (4,)


## 5. Evaluation

**Accuracy** = the fraction of test titles whose predicted class is right
(by chance, with four balanced classes: 0.25). Per class, **precision** is
how often a prediction of that class is right, **recall** how many titles
of that class are found.

In [8]:
predicted = classifier.predict(X_test)
print("accuracy on the training titles:", round(accuracy_score(train.label, classifier.predict(X_train)), 3))
print("accuracy on the test titles:    ", round(accuracy_score(test.label, predicted), 3), "\n")
print(classification_report(test.label, predicted, digits=3))

accuracy on the training titles: 0.999
accuracy on the test titles:     0.864 

                precision    recall  f1-score   support

cardiovascular      0.871     0.880     0.876       200
     digestive      0.867     0.880     0.873       200
     neoplasms      0.886     0.855     0.870       200
       nervous      0.832     0.840     0.836       200

      accuracy                          0.864       800
     macro avg      0.864     0.864     0.864       800
  weighted avg      0.864     0.864     0.864       800



The **confusion matrix**: rows are the true classes, columns the predicted
ones; the diagonal holds the correct answers.

In [9]:
pd.crosstab(test.label, predicted, rownames=["true"], colnames=["predicted"])

predicted,cardiovascular,digestive,neoplasms,nervous
true,,,,
cardiovascular,176,7,4,13
digestive,8,176,8,8
neoplasms,6,10,171,13
nervous,12,10,10,168


Some test titles the classifier got wrong:

In [10]:
errors = test.assign(predicted=predicted)[lambda d: d.label != d.predicted]
errors.sample(8, random_state=0)[["title", "label", "predicted"]]

,title,label,predicted
1471,Molecular genetic basis of maple syrup urine disease in a family with two defective alleles for branched chain acyltransferase and localization of the gene to human chromosome 1.,nervous,digestive
4856,Pulmonary acinar development in diaphragmatic hernia.,digestive,nervous
661,Clinical management of the complications of HIV infection.,digestive,cardiovascular
270,Circulating CD8+ cytotoxic T lymphocytes specific for HTLV-I pX in patients with HTLV-I associated neurological disease.,neoplasms,nervous
69,The clinical utility of carotid duplex scanning.,cardiovascular,nervous
649,Probability of stroke: a risk profile from the Framingham Study.,nervous,cardiovascular
3510,Thyroid disease in the elderly.,neoplasms,nervous
2888,Experimental isobaric subarachnoid hemorrhage: regional mitochondrial function during the acute and late phase.,cardiovascular,nervous


## 6. How many training titles are needed?

The same training, with only *n* titles per class (and always the same
test titles).

In [11]:
print(" n per class   accuracy")
for n in [1, 2, 5, 10, 25, 50, 100, 200, 400]:
    subset = train.groupby("label").sample(n, random_state=0)
    small = LogisticRegression(max_iter=1000).fit(embed(subset.title.tolist()), subset.label)
    print(f"{n:12d}   {accuracy_score(test.label, small.predict(X_test)):.3f}")

 n per class   accuracy


           1   0.520


           2   0.596


           5   0.748


          10   0.797


          25   0.823


          50   0.840


         100   0.848


         200   0.865


         400   0.865


## 7. Routing the MultiCaRe texts

The trained classifier, applied to the four texts of the embedding
notebook — which are not titles, and not from PubMed abstracts but from
case reports. One row per text, one probability per class:

In [12]:
X_cases = embed(list(TEXTS.values()))
probabilities = pd.DataFrame(classifier.predict_proba(X_cases),
                             index=TEXTS.keys(), columns=classifier.classes_).round(3)
probabilities["predicted"] = classifier.predict(X_cases)
probabilities

,cardiovascular,digestive,neoplasms,nervous,predicted
Text 1,0.000,0.999,0.001,0.000,digestive
Text 2,0.000,0.997,0.000,0.003,digestive
Text 3,0.016,0.272,0.002,0.711,nervous
Text 4,0.000,0.971,0.013,0.016,digestive


## What to notice

* **The model was not trained at all.** Only the logistic regression
  learned: 4 × 384 weights and 4 biases. Every title was turned into its
  `[CLS]` vector once, by the same frozen model as in the embedding notebook.
* **Padding is harmless.** `Diarrhea.` gets `[PAD]` tokens (mask 0) so its
  row is as long as the other one, and text 1 gives the same `[CLS]` vector
  alone or in a batch (largest difference ≈ 7 × 10⁻⁷, rounding).
* **0.864 on unseen titles**, against 0.25 by chance, with every class
  between 0.84 and 0.88 (F1). On the training titles it is 0.999: the
  difference is why accuracy is always measured on titles kept apart.
* **The mistakes are often arguable.** *Probability of stroke* is labeled
  nervous and predicted cardiovascular; a stroke is both. Some labels are
  surprising in themselves (*Thyroid disease in the elderly* is labeled
  neoplasms). A title is a short text, and the corpus labels the whole
  abstract.
* **A few examples go a long way.** With 5 titles per class the accuracy is
  already 0.748, with 50 it is 0.840; from 200 to 400 it does not move
  (0.865). The embedding already places titles by topic; the classifier
  only has to draw the borders between the groups.
* **It transfers to case reports.** Texts 1 and 2 go to *digestive* with
  P ≈ 1.0, text 3 to *nervous* — but with 0.711, and 0.272 for
  *digestive*: a confused patient found on the floor is less clear-cut than
  abdominal pain. The classifier never saw a case report; the texts are
  longer than titles and describe one patient instead of a study.
* **It always answers one of its four classes.** Text 4 — a rash with lip
  and eye swelling, a skin or allergic reaction — fits none of them, yet it
  goes to *digestive* with P = 0.971. The probabilities are shared among
  the classes the classifier knows and always add up to 1: a high
  probability means "more like this class than like the other three", not
  "surely this class". Handling *none of these* needs a fifth class with
  its own examples, or a threshold on similarity.
* **Try it:** set `MODEL_NAME` to
  `"microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"` (a
  biomedical model whose `[CLS]` was not trained to summarize a text) and
  compare the accuracy; or change `PER_CLASS` and the training sizes.

## Credits

* Data: T. Schopf, D. Braun, F. Matthes. *Evaluating Unsupervised Text
  Classification: Zero-shot and Similarity-based Approaches.* NLPIR 2022.
  [Medical Abstracts TC Corpus](https://github.com/sebischair/Medical-Abstracts-TC-Corpus),
  CC BY-SA 3.0.
* Model: [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5).
* The four texts: MultiCaRe cases `PMC5137649_01`, `PMC11722600_01`,
  `PMC6949661_01`, `PMC6134951_01` (sample in `../../data/multicare/data/csv_sample/`).